# Ex02 — Hour 2: Alert correlation + Root Cause Analysis

**Goal:** Given a cascade of alerts across services, find the ROOT cause — not just the LOUDEST service.

**Scenarios in scope (Hour 2):** S04 (TLS cert), S05 (JVM leak), S06 (T24 cascade), S07 (silent OOM).

**What you'll do:**
1. Group raw alerts by fingerprint + time window
2. Build the affected-service subgraph from topology
3. Run 3 RCA rankers: PageRank, earliest-drift, drift-count
4. Fuse via Weighted Reciprocal Rank Fusion (RRF)
5. Validate against the scenario's expected root cause


In [ ]:
# Auto-detect and install missing required libraries in the current Jupyter kernel
import sys
import subprocess

required_libs = ["pandas", "matplotlib", "networkx", "numpy", "statsmodels", "scikit-learn"]
missing_libs = []

for lib in required_libs:
    try:
        # Handle mapping from import name to pip package name if different
        import_name = "sklearn" if lib == "scikit-learn" else lib
        __import__(import_name)
    except ImportError:
        missing_libs.append(lib)

if missing_libs:
    print(f"Missing libraries: {missing_libs}. Installing...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install"] + missing_libs)
        print("Successfully installed missing libraries!")
    except Exception as e:
        print(f"Error installing libraries: {e}")
else:
    print("All required libraries are already installed.")


In [ ]:
import sqlite3, json, importlib.util
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Connect to workshop database
for path in [Path("workshop.db"), Path("../workshop.db")]:
    if path.exists():
        DB = path.resolve()
        break
else:
    DB = (Path.cwd().parent / "workshop.db").resolve()
conn = sqlite3.connect(DB)

# Helper to import modules with kebab-case filenames (workshop convention)
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

print(f"DB: {DB} ({DB.stat().st_size//1024} KB)")
print(f"Tables: {[r[0] for r in conn.execute('SELECT name FROM sqlite_master WHERE type=\"table\"').fetchall()]}")


## 1. Pick a scenario — S06: T24 cascade (4-hop)

Look at the alert sequence:

In [ ]:
alerts = pd.read_sql("SELECT opened_at, severity, service, rule_id, phase FROM alerts "
                     "WHERE scenario = 'S06' ORDER BY opened_at", conn)
alerts['opened_at'] = pd.to_datetime(alerts['opened_at'])
alerts


**Observation**: alerts fire in the order `t24-service` → `esb` → `datapower` → `bb-edge` over 6 minutes.
Naive "fix the alerting service" would page on the FIRST alerting service. Let's see if RCA agrees with that vs the actual root cause.

## 2. Build cascade subgraph from topology

In [ ]:
import networkx as nx
edges = pd.read_sql("SELECT src_service, dst_service FROM topology", conn)
G = nx.DiGraph()
for _, r in edges.iterrows(): G.add_edge(r['src_service'], r['dst_service'])

affected = set(alerts['service'])
# Subgraph: only services + 1-hop neighbors of affected
sub_nodes = set(affected)
for s in affected:
    if s in G:
        sub_nodes.update(G.predecessors(s)); sub_nodes.update(G.successors(s))
sub = G.subgraph(sub_nodes)
print(f"Affected: {affected}")
print(f"Subgraph nodes: {sub.nodes()}")
print(f"Subgraph edges: {list(sub.edges())}")


## 3. Run PageRank (R1) on reverse topology

Why reverse? Because "blame flows from caller TO callee". A → B means A depends on B; if B fails, A degrades. So reverse the edges and let PageRank accumulate mass at services that many alerting services depend on.

In [ ]:
Gr = G.reverse()
pers = {n: 1.0 for n in Gr.nodes()}
for s in affected: pers[s] = 10.0
pr = nx.pagerank(Gr, personalization=pers, max_iter=100)
top5_pr = sorted(pr.items(), key=lambda kv: -kv[1])[:5]
for s, v in top5_pr: print(f"  {s:30s} {v:.4f}")


## 4. Run earliest-drift ranker (R2)

For each service, find when ANY of its key metrics first deviated > 3σ from baseline.
Service that drifted EARLIEST = most likely upstream root cause.

In [ ]:
import numpy as np
from collections import defaultdict
metrics = pd.read_sql("SELECT timestamp, service, metric, value FROM metrics "
                      "WHERE scenario = 'S06' ORDER BY timestamp", conn)
earliest = {}
for (svc, m), g in metrics.groupby(['service', 'metric']):
    vals = g['value'].values
    if len(vals) < 70: continue
    mu, sigma = vals[:60].mean(), max(vals[:60].std(), 1e-6)
    drift_idx = np.where(np.abs(vals[60:] - mu) > 3 * sigma)[0]
    if len(drift_idx) == 0: continue
    ts = g.iloc[60 + drift_idx[0]]['timestamp']
    if svc not in earliest or ts < earliest[svc]:
        earliest[svc] = ts

for s, ts in sorted(earliest.items(), key=lambda kv: kv[1])[:5]:
    print(f"  {s:30s} first drift @ {ts}")


## 5. Fuse with Weighted RRF

Combine the rankers. Earliest-drift gets highest weight (0.5) because it's the most directional signal in cascade scenarios.

In [ ]:
def rrf(rankings, weights, k=60):
    score = defaultdict(float)
    for r, w in zip(rankings, weights):
        for i, svc in enumerate(r):
            score[svc] += w / (k + i + 1)
    return sorted(score.items(), key=lambda kv: -kv[1])

r1 = [s for s, _ in top5_pr]
r2 = [s for s, _ in sorted(earliest.items(), key=lambda kv: kv[1])]
# R3 — count of drifted metrics per service
drift_cnt = defaultdict(int)
for (svc, m), g in metrics.groupby(['service', 'metric']):
    vals = g['value'].values
    if len(vals) < 70: continue
    mu, sigma = vals[:60].mean(), max(vals[:60].std(), 1e-6)
    if max(np.abs(vals[60:] - mu)) > 3 * sigma:
        drift_cnt[svc] += 1
r3 = [s for s, _ in sorted(drift_cnt.items(), key=lambda kv: -kv[1])]

fused = rrf([r1, r2, r3], [0.3, 0.5, 0.2])
print("=== Fused root-cause ranking ===")
for s, v in fused[:5]: print(f"  {s:30s} {v:.4f}")


## 6. Validate against expected

In [ ]:
sc = json.loads(conn.execute("SELECT full_json FROM scenarios WHERE id='S06'").fetchone()[0])
expected = sc['expected_rca']
fused_top = fused[0][0]
print(f"Expected root: {expected['top_service']}")
print(f"Fused #1:      {fused_top}")
print(f"Match: {fused_top == expected['top_service']}")
print(f"\nExplanation: {expected['explanation']}")


## 7. Tie-breaker R4: Granger causality
R1 (PageRank), R2 (earliest-drift), R3 (drift count) are all heuristics. When they disagree on tie, you need an *orthogonal* signal: **does service A's metric statistically lead service B's in time?**

Granger causality tests whether past values of X help predict future values of Y, beyond Y's own past. It is **not** "true" causality — only predictive precedence — but for ranking a root cause that's often what you need.

> Note: Granger is *one tool* within the broader causal-inference family (Pearl, do-calculus, structural causal models). We pick Granger here as a *cheap tie-breaker*, not as the foundation of RCA. As you'll see below, it can also be **misleading on short, heavy-tail series** — exactly why fusion matters.


In [ ]:
# Granger helper: granger_lead(x, y) returns "does x lead y?" — small p = strong yes.
from statsmodels.tsa.stattools import grangercausalitytests
import warnings; warnings.filterwarnings('ignore')

def granger_lead(x, y, max_lag=2):
    """statsmodels expects column 0 = predicted variable, column 1 = candidate cause.
       Returns (best_p_value, best_lag_in_samples)."""
    data = np.column_stack([y, x])
    try:
        res = grangercausalitytests(data, maxlag=max_lag, verbose=False)
        best = min(res.items(), key=lambda kv: kv[1][0]['ssr_ftest'][1])
        return best[1][0]['ssr_ftest'][1], best[0]
    except Exception:
        return 1.0, 0

# Use latency_p99_ms — same metric for all 4 alerting services so pairs are comparable.
SVCS = ['t24-service', 'esb', 'datapower', 'bb-edge']
series = {}
for s in SVCS:
    df = pd.read_sql("""SELECT value FROM metrics
                       WHERE scenario='S06' AND service=? AND metric='latency_p99_ms'
                       ORDER BY timestamp""", conn, params=[s])
    series[s] = df['value'].values
print(f"Loaded {len(series)} services × {len(next(iter(series.values())))} samples (latency_p99_ms).")


## 8. Run Granger on every directed pair
For each ordered pair (caller, callee), test whether *caller's latency Granger-causes callee's*. Small p = caller leads callee.


In [ ]:
rows = []
for caller in SVCS:
    for callee in SVCS:
        if caller == callee: continue
        p, lag = granger_lead(series[caller], series[callee])
        rows.append({'caller': caller, 'callee': callee, 'p_value': round(p, 4), 'best_lag_samples': lag})
gdf = pd.DataFrame(rows).sort_values('p_value')
print(gdf.head(8).to_string(index=False))


## 9. Build R4 — Granger lead score
A service leading many others (low p-values) gets a high score. We aggregate: `score(S) = sum(-log10(p))` over all pairs where S is the caller. Higher = stronger leading signal.


In [ ]:
r4_raw = {}
for s in SVCS:
    mask = (gdf['caller'] == s) & (gdf['p_value'] < 0.5)
    r4_raw[s] = float((-np.log10(gdf.loc[mask, 'p_value'].clip(lower=1e-6))).sum())

r4_sorted = sorted(r4_raw.items(), key=lambda kv: -kv[1])
r4 = [s for s, _ in r4_sorted]

print("R4 (Granger lead score):")
for s, sc in r4_sorted:
    print(f"  {s:30s} {sc:.3f}")

print("\n>> Notice: R4 #1 is probably NOT t24-service. Granger picks the wrong service here.")
print(">> Why? p99 latency is heavy-tail + short series (105 samples). Bursty downstream spikes")
print(">> auto-predict earlier upstream values via noise correlation. This is a known Granger pitfall.")


## 10. Your turn — fuse R1 + R2 + R3 + R4
You now have 4 rankers, and R4 just gave the wrong answer. **Question for you**: can fusion still pick the correct root cause when one ranker fails?

Use the `rrf()` helper with weights `[0.2, 0.5, 0.1, 0.2]`:
- R2 (earliest-drift) gets the **highest** weight — timing is the most direct evidence
- R4 (Granger) gets a **modest** weight because it can be unreliable

**Complete the cell, then verify**: does the fused ranking still match `expected['top_service']`?


In [ ]:
# EXERCISE: fuse the four rankers using rrf(rankings, weights, k=60).
# Existing rankings: r1, r2, r3, r4 — all are lists of services sorted top-first.
fused4 = rrf([r1, r2, r3, r4], [0.2, 0.5, 0.1, 0.2], k=60)   # completed

if fused4 is not None:
    print('=== 4-way fused ranking ===')
    for s, v in fused4[:5]:
        print(f'  {s:30s} {v:.4f}')
    correct = fused4[0][0] == expected['top_service']
    print(f"\nExpected: {expected['top_service']} | 4-way #1: {fused4[0][0]} | Match: {correct}")
    if correct:
        print("\n>> Fusion compensated for R4's mistake. This is why production stacks fuse 3-5 rankers.")


### Follow-up — what happens if R4 gets too much weight?
Try changing the weights from `[0.2, 0.5, 0.1, 0.2]` to `[0.1, 0.2, 0.1, 0.6]` (heavy R4). Re-run. Does the fused #1 still match `t24-service`?

This is the **weight-tuning lesson**: the same fusion can succeed or fail depending on how much you trust each individual ranker. In production, weights are tuned on past labelled incidents.


## 11. Visualize — 4 rankers side-by-side
Where each ranker agrees on the root cause and where they disagree.


In [ ]:
import matplotlib.pyplot as plt

def to_rank_score(ranking_list, top_n=4):
    return {s: (top_n - i - 1) / max(top_n - 1, 1) for i, s in enumerate(ranking_list[:top_n])}

r1_n = to_rank_score(r1); r2_n = to_rank_score(r2)
r3_n = to_rank_score(r3); r4_n = to_rank_score(r4)

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(SVCS)); w = 0.2
ax.bar(x - 1.5*w, [r1_n.get(s, 0) for s in SVCS], w, label='R1 PageRank',      color='#3b82f6')
ax.bar(x - 0.5*w, [r2_n.get(s, 0) for s in SVCS], w, label='R2 Earliest-drift', color='#22c55e')
ax.bar(x + 0.5*w, [r3_n.get(s, 0) for s in SVCS], w, label='R3 Drift count',    color='#a855f7')
ax.bar(x + 1.5*w, [r4_n.get(s, 0) for s in SVCS], w, label='R4 Granger',        color='#f59e0b')
ax.set_xticks(x); ax.set_xticklabels(SVCS, rotation=15)
ax.set_ylabel('Normalized rank (top=1)')
ax.set_title('S06 RCA — four rankers compared')
ax.legend(loc='upper right', fontsize=9); plt.tight_layout(); plt.show()

print("\n>> Read this chart: where all 4 rankers cluster on the same service, you have high")
print(">> confidence. Where they disagree (e.g. t24-service: tall green, short orange), one")
print(">> ranker is leading and others lag. Fusion smooths these disagreements.")


## 7. Now run RCA for S07 (silent OOM cascade)

S07 has ONLY 1 alert (k8s-events warning) — RCA must use METRIC DRIFT not alerts.

**EXERCISE**: apply the same pipeline to S07. Note: only `cache_hit_rate` and `stale_response_count` will drift.
What does PageRank tell you? What does earliest-drift tell you? Which is right?

In [ ]:
# EXERCISE: copy-adapt the pipeline above to scenario = 'S07'
# Expected root: bb-confirmation-service (only metric that drifts: pod_restart_count)

# 1. Fetch metrics for S07
metrics_s07 = pd.read_sql("SELECT timestamp, service, metric, value FROM metrics "
                          "WHERE scenario = 'S07' ORDER BY timestamp", conn)

# 2. Run earliest-drift ranker (R2) for S07. We use 4 * sigma to filter out latency noise.
earliest_s07 = {}
for (svc, m), g in metrics_s07.groupby(['service', 'metric']):
    vals = g['value'].values
    if len(vals) < 70: continue
    mu, sigma = vals[:60].mean(), max(vals[:60].std(), 1e-6)
    drift_idx = np.where(np.abs(vals[60:] - mu) > 4 * sigma)[0]
    if len(drift_idx) == 0: continue
    ts = g.iloc[60 + drift_idx[0]]['timestamp']
    if svc not in earliest_s07 or ts < earliest_s07[svc]:
        earliest_s07[svc] = ts

print("S07 Earliest drift ranking:")
sorted_earliest_s07 = sorted(earliest_s07.items(), key=lambda kv: kv[1])
for s, ts in sorted_earliest_s07[:5]:
    print(f"  {s:30s} first drift @ {ts}")

# 3. Verify top service
if sorted_earliest_s07:
    top_svc_s07 = sorted_earliest_s07[0][0]
    print(f"\nExpected: bb-confirmation-service | RCA Top: {top_svc_s07} | Match: {top_svc_s07 == 'bb-confirmation-service'}")


## 8. Discussion

1. **PageRank alone for S07 fails** — only 1 alerting service. Why is earliest-drift better here?
2. For S04 (TLS cert), the cascade is REVERSED (entrypoint failure, no upstream). What does PageRank rank? What does drift rank? Do they agree?
3. **Decision rule**: if RCA confidence < 0.5 (top score / sum scores), page a human instead of auto-remediate. Implement this threshold check.
